# 01 — EDA (Home Credit Default Risk)

**Objetivo:** caracterizar en profundidad `application_*`: `TARGET` y baselines ingenuos, tipos y cardinalidad, nulos globales y **nulos por clase `TARGET`**, alineación train/test, **deriva (KS)** en numéricas, IDs, volumetría de tablas auxiliares, sanity checks y percentiles, correlaciones y **heatmap**, categóricas (**Cramér V**), tasas de default por grupo, **EXT_SOURCE** por clase, y **bureau** (filas por cliente vs default). Modelado en notebooks posteriores.

**Problema:** predecir incumplimiento (`TARGET`=1). Métrica Kaggle: **ROC-AUC**.

**Cómo está documentado:** debajo de cada sección temática, hay una celda **"Documentación — …"** justo **antes** del código asociado; resume qué hace ese bloque, qué objetos usa y qué debes interpretar en las salidas o gráficos.

**Sobre salidas (outputs) y acceso:** si ejecutas el notebook y **guardas** el archivo `.ipynb`, tablas y figuras quedan embebidas en el JSON del notebook. Cualquier herramienta que **lea ese archivo** (incluido un asistente con acceso al workspace) puede ver esas salidas **tal como están guardadas**. No hay acceso a la ejecución en vivo ni a resultados que no estén guardados en el archivo.


## Fuentes de datos

| Recurso | Uso |
|--------|-----|
| `data/raw/*.csv` | CSV de la competición |
| `configs/home_credit.yaml` | Archivos, claves, `TARGET` |
| `configs/base.yaml` | Semilla, splits |
| `HomeCredit_columns_description.csv` | Diccionario de columnas |

Código: `src.data.home_credit`.


### Documentación — Configuración e imports

- **`sys.path` / `ROOT`**: permite importar el paquete `src` aunque el kernel se ejecute desde `notebooks/` o la raíz del repo.
- **Librerías**: `numpy`, `pandas`, `matplotlib`, `yaml`; **`scipy.stats`**: `chi2_contingency` (tablas de contingencia) y `ks_2samp` (comparar distribuciones train vs test).
- **`IPython.display.display`**: muestra tablas en el notebook.
- **`src.data.home_credit`**: lectura de CSV y rutas según `configs/home_credit.yaml`.
- **`src.data.quality`**: funciones de calidad (`missing_summary`, `basic_range_checks`).
- **`set_seed(42)`**: alinea con `configs/base.yaml` para muestreos reproducibles en celdas posteriores.
- **Salida**: imprime la raíz del proyecto resuelta por `project_root()`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import chi2_contingency, ks_2samp # test Kolmogorov-Smirnov

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display

from src.data.home_credit import keys, read_table, table_path, target_column
from src.data.quality import basic_range_checks, missing_summary
from src.utils.paths import project_root
from src.utils.seed import set_seed

plt.style.use("ggplot")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

set_seed(42)
print("project_root:", project_root())


### Documentación — Carga de `application_train` / `application_test`

- **`read_table`**: lee los CSV completos desde `data/raw/` (rutas definidas en config).
- **`y_col`**: nombre de la columna objetivo (`TARGET`), solo presente en train.
- **`k_curr`**: nombre de la clave primaria (`SK_ID_CURR`) para joins y comprobaciones.
- **Salida**: dimensiones de ambos DataFrames y comprobación de que `TARGET` no está en test.


In [ ]:
train = read_table("application_train")
test = read_table("application_test")

y_col = target_column()
k_curr = keys()["sk_id_curr"]

print("train:", train.shape, "| test:", test.shape)
print("TARGET en train:", y_col in train.columns)
print("TARGET en test:", y_col in test.columns)


## Tabla principal `application_train` / `application_test`

- Una fila por solicitud (`SK_ID_CURR`).
- `TARGET` solo en train.
- Numéricas, categóricas y flags.


### Documentación — Distribución del `TARGET` y baselines

- **`value_counts`**: recuentos de clases 0 y 1.
- **Métricas**: proporción de positivos; ratio aproximado entre mayoritaria y minoritaria.
- **Baselines**: accuracy que obtendrías prediciendo siempre 0 o siempre 1 (sin modelo), para contextualizar ROC-AUC posterior.
- **Figuras**: barras de conteos y gráfico circular de proporciones.


In [ ]:
vc = train[y_col].value_counts().sort_index()
n0, n1 = int(vc.get(0, 0)), int(vc.get(1, 0))
imbalance = n1 / len(train) if len(train) else 0
ratio_neg_pos = n0 / n1 if n1 else np.nan

print("Distribución TARGET:", vc.to_string())
print(f"Proporción positivos (TARGET=1): {imbalance:.4f}")
print(f"Ratio clases 0:1 (aprox.): {ratio_neg_pos:.2f}:1")
print("Baseline siempre 0: accuracy ~", f"{1 - imbalance:.4f}", "(mayoría); P/R clase 1 = 0")
print("Baseline siempre 1: accuracy ~", f"{imbalance:.4f}", "| recall 1 = 1 | precision ~", f"{imbalance:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
vc.plot(kind="bar", ax=axes[0], color=["#4c78a8", "#f58518"])
axes[0].set_xlabel("TARGET")
axes[0].set_ylabel("Recuento")
axes[0].set_title("Conteos (train)")
axes[1].pie(
    [n0, n1],
    labels=["TARGET=0", "TARGET=1"],
    autopct="%1.1f%%",
    colors=["#4c78a8", "#f58518"],
    startangle=90,
)
axes[1].set_title("Proporción")
plt.tight_layout()


### Tipos y columnas

Dtypes, cardinalidad `object`, flags binarios, columnas casi constantes.


### Documentación — Tipos de datos y cardinalidad

- **`dtypes`**: conteo de tipos por columna en train.
- **`feat_cols`**: todas las columnas excepto `TARGET`.
- **`num_feats` / `obj_cols`**: separación numéricas vs texto (categóricas crudas).
- **Binarias**: columnas numéricas que solo toman 0 y 1 (flags).
- **`card`**: cardinalidad de columnas `object` (orden descendente).
- **Constantes**: columnas con un único valor (poca información marginal).


In [ ]:
dtypes_train = train.dtypes.value_counts()
print("Dtypes en train:")
print(dtypes_train)

feat_cols = [c for c in train.columns if c != y_col]
num_feats = train[feat_cols].select_dtypes(include=[np.number]).columns.tolist()
obj_cols = train[feat_cols].select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", len(num_feats), "| Object:", len(obj_cols))

binary_like = []
for c in num_feats:
    u = train[c].dropna().unique()
    if len(u) <= 2:
        ufloat = {float(x) for x in u}
        if ufloat <= {0.0, 1.0}:
            binary_like.append(c)
print("Columnas numéricas binarias (solo 0/1):", len(binary_like))

card = train[obj_cols].nunique(dropna=False).sort_values(ascending=False)
print("Top 15 por cardinalidad (object):")
print(card.head(15))

nunique_all = train[feat_cols].nunique(dropna=False)
constant_cols = nunique_all[nunique_all <= 1].index.tolist()
print("Columnas con <=1 valor distinto:", len(constant_cols), constant_cols[:15])


## Valores faltantes

Fracción de NA global y comparación **TARGET=0 vs TARGET=1** en las columnas con más nulos.


### Documentación — Valores faltantes

- **`missing_summary`**: fracción de `NA` por columna (todo el train).
- **Visualización**: barras horizontales de las 25 columnas con más nulos.
- **Bloque siguiente**: para las 30 columnas con más nulos, compara la **tasa de NA** entre filas con `TARGET=0` y `TARGET=1`; si difiere, el patrón de missing puede ser informativo (MNAR / necesidad de indicadores).


In [ ]:
miss = missing_summary(train).sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f"Columnas con al menos un nulo: {len(miss_nonzero)} / {len(train.columns)}")
display(miss_nonzero.head(25).to_frame("frac_na"))

fig, ax = plt.subplots(figsize=(8, 5))
miss_nonzero.head(25).plot(kind="barh", ax=ax, legend=False, color="#4c78a8")
ax.set_xlabel("Fracción NA")
ax.set_title("Top 25 columnas por % de nulos (train)")
plt.tight_layout()

cols_miss = miss_nonzero.head(30).index.tolist()
rows = []
for col in cols_miss:
    if col == y_col:
        continue
    m0 = train.loc[train[y_col] == 0, col].isna().mean()
    m1 = train.loc[train[y_col] == 1, col].isna().mean()
    rows.append({"column": col, "na_rate_t0": m0, "na_rate_t1": m1, "diff": abs(m1 - m0)})
miss_target = pd.DataFrame(rows).sort_values("diff", ascending=False)
print("Top 15 columnas donde la tasa de NA difiere más entre TARGET 0 y 1:")
display(miss_target.head(15))


## Calidad: duplicados e IDs

`SK_ID_CURR` único en train y en test; sin solapamiento train∩test.


### Documentación — Duplicados e IDs

- **`duplicated()`** sobre `SK_ID_CURR`: debe ser 0 en train y test (un registro por solicitud).
- **Intersección de sets** train vs test: no debe haber IDs compartidos entre archivos (simula el holdout de la competición).


In [ ]:
dup_train = train[k_curr].duplicated().sum()
dup_test = test[k_curr].duplicated().sum()
overlap = set(train[k_curr]) & set(test[k_curr])

print(f"Filas duplicadas por {k_curr} en train: {dup_train}")
print(f"Filas duplicadas por {k_curr} en test: {dup_test}")
print(f"IDs compartidos train∩test: {len(overlap)}")


### Alineación train / test

Columnas exclusivas de cada tabla y dtypes en el cruce común (sin `TARGET`).


### Documentación — Alineación de columnas train / test

- **Conjuntos de nombres**: columnas solo en train (p. ej. `TARGET`), solo en test, y comunes.
- **Dtypes**: lista columnas comunes donde `train` y `test` declaran tipos distintos (riesgo al concatenar o al aplicar el mismo pipeline).


In [ ]:
only_train = set(train.columns) - set(test.columns)
only_test = set(test.columns) - set(train.columns)
common = (set(train.columns) & set(test.columns)) - {y_col}

print("Solo en train:", only_train)
print("Solo en test:", only_test)
print(f"Columnas comunes (sin TARGET): {len(common)}")

dtype_mismatch = []
for col in sorted(common):
    if train[col].dtype != test[col].dtype:
        dtype_mismatch.append((col, train[col].dtype, test[col].dtype))
print(f"Diferencias de dtype en comunes: {len(dtype_mismatch)}")
if dtype_mismatch:
    display(pd.DataFrame(dtype_mismatch, columns=["column", "dtype_train", "dtype_test"]).head(25))


### Coherencia train vs test (exploratorio)

Medias en numéricas comunes y test **Kolmogorov-Smirnov** (muestra hasta 50k filas por serie). *p* bajo sugiere distribuciones distintas.


### Documentación — Coherencia de distribuciones (train vs test)

- **`num_common`**: columnas numéricas presentes en ambos conjuntos.
- **Medias**: comparación rápida de `mean_train` vs `mean_test` y diferencia relativa.
- **Kolmogorov–Smirnov (`ks_2samp`)**: contrasta si dos muestras vienen de la misma distribución; aquí se submuestrea hasta 50k filas por lado si hace falta por coste.
- **Interpretación**: p-valores muy bajos o estadísticos KS altos sugieren **deriva** entre train y test en esa variable.


In [ ]:
num_common = [
    col
    for col in common
    if col in train.columns
    and pd.api.types.is_numeric_dtype(train[col])
    and pd.api.types.is_numeric_dtype(test[col])
]
rows = []
for col in num_common:
    t_mean = train[col].mean()
    s_mean = test[col].mean()
    diff_rel = abs(t_mean - s_mean) / (abs(t_mean) + 1e-12)
    sub_t = train[col].dropna()
    sub_s = test[col].dropna()
    if len(sub_t) > 50000:
        sub_t = sub_t.sample(50000, random_state=42)
    if len(sub_s) > 50000:
        sub_s = sub_s.sample(50000, random_state=42)
    if len(sub_t) < 2 or len(sub_s) < 2:
        continue
    ks_stat, ks_p = ks_2samp(sub_t, sub_s)
    rows.append(
        {
            "column": col,
            "mean_train": t_mean,
            "mean_test": s_mean,
            "rel_mean_diff": diff_rel,
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_p,
        }
    )
drift = pd.DataFrame(rows).sort_values("ks_statistic", ascending=False)
print("Top 15 por estadístico KS:")
display(drift.head(15))
print("Top 10 por divergencia relativa de medias:")
display(drift.sort_values("rel_mean_diff", ascending=False).head(10)[["column", "mean_train", "mean_test", "rel_mean_diff"]])


## Otras tablas (volumen)

Filas por CSV auxiliar (comportamiento = múltiples filas por cliente).


### Documentación — Volumetría de tablas auxiliares

- **`count_csv_rows`**: cuenta líneas sin cargar el CSV completo en memoria (solo lectura byte a byte).
- **Uso**: dimensionar el trabajo de joins y agregaciones en `bureau`, balances, cuotas, etc.


In [ ]:
def count_csv_rows(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f) - 1


aux_keys = [
    "bureau",
    "bureau_balance",
    "previous_application",
    "pos_cash_balance",
    "installments_payments",
    "credit_card_balance",
]
rows = {}
for k in aux_keys:
    rows[k] = count_csv_rows(table_path(k))

pd.Series(rows, name="n_rows").sort_values(ascending=False)


## Rangos plausibles (sanity checks)

`basic_range_checks` + percentiles de `DAYS_BIRTH`. Rango `DAYS_BIRTH` ampliado a (-30000, 0).


### Documentación — Sanity checks de rangos

- **`range_hints`**: intervalos plausibles para scores externos (`EXT_SOURCE_*`), días (`DAYS_BIRTH`, `DAYS_EMPLOYED` con placeholder 365243).
- **`basic_range_checks`**: comprueba que valores no nulos caen en el rango (o marca REVISAR).
- **Percentiles de `DAYS_BIRTH`**: contexto si el check rígido falla por valores límite.


In [ ]:
range_hints = {
    "EXT_SOURCE_1": (0.0, 1.0),
    "EXT_SOURCE_2": (0.0, 1.0),
    "EXT_SOURCE_3": (0.0, 1.0),
    "DAYS_BIRTH": (-30000, 0),
    "DAYS_EMPLOYED": (-25000, 365243),
}

present = {k: v for k, v in range_hints.items() if k in train.columns}
chk = basic_range_checks(train, present)
for col, ok in sorted(chk.items()):
    print(f"{col}: {'OK' if ok else 'REVISAR'}")

if "DAYS_BIRTH" in train.columns:
    db = train["DAYS_BIRTH"].dropna()
    print("DAYS_BIRTH min/max/p01/p99:", float(db.min()), float(db.max()), float(db.quantile(0.01)), float(db.quantile(0.99)))

if "DAYS_EMPLOYED" in train.columns:
    special = (train["DAYS_EMPLOYED"] == 365243).mean()
    print(f"Fracción DAYS_EMPLOYED == 365243 (placeholder): {special:.4f}")


## Asociación con `TARGET`

Pearson en numéricas; heatmap entre top correlaciones; **Cramér V** y chi² en `object`; tasas de default por grupo.


### Documentación — Asociación con `TARGET` (numéricas y categóricas)

- **Pearson**: correlación lineal de cada numérica con `TARGET`; ordena por valor absoluto.
- **Heatmap**: matriz de correlación entre las **top 25** variables por |corr| con `TARGET` más la propia `TARGET` (dependencias lineales entre candidatos a feature).
- **Cramér V**: a partir de `chi2_contingency` sobre tablas cruzadas categoría × `TARGET`; mide asociación 0–1 en categóricas.
- **Tablas por grupo**: tasa de default (`mean` de `TARGET`) y volumen `n` para variables de negocio (`CODE_GENDER`, etc.).


In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if y_col in num_cols:
    num_cols.remove(y_col)

corr = train[num_cols + [y_col]].corr(numeric_only=True)[y_col].drop(labels=[y_col], errors="ignore")
corr = corr.replace([np.inf, -np.inf], np.nan).dropna()
corr_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)

print("Top 15 correlación con TARGET (Pearson):")
display(corr_sorted.head(15).to_frame("corr_with_TARGET"))
print("Bottom 5 (menor |corr|):")
display(corr_sorted.tail(5).to_frame("corr_with_TARGET"))

top_k = 25
top_for_heatmap = [c for c in corr_sorted.head(top_k).index if c in train.columns]
sub = train[top_for_heatmap + [y_col]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sub.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sub.columns)))
ax.set_yticks(range(len(sub.columns)))
ax.set_xticklabels(sub.columns, rotation=90, fontsize=7)
ax.set_yticklabels(sub.columns, fontsize=7)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f"Correlaciones: top {top_k} por |corr| con TARGET + TARGET")
plt.tight_layout()


def cramers_v(conf: np.ndarray) -> float:
    chi2, _, _, _ = chi2_contingency(conf, correction=False)
    n = conf.sum()
    r, k = conf.shape
    return float(np.sqrt(chi2 / (n * (min(r, k) - 1))))


cat_rows = []
for col in obj_cols:
    tab = pd.crosstab(train[col].fillna("__NA__"), train[y_col])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        continue
    try:
        chi2, p, _, _ = chi2_contingency(tab.values)
        cv = cramers_v(tab.values.astype(float))
        cat_rows.append({"column": col, "chi2_pvalue": p, "cramers_v": cv})
    except ValueError:
        continue
cat_assoc = pd.DataFrame(cat_rows).sort_values("cramers_v", ascending=False)
print("Asociación categórica vs TARGET (Cramér V):")
display(cat_assoc)

for col in ["CODE_GENDER", "NAME_INCOME_TYPE", "FLAG_OWN_CAR"]:
    if col not in train.columns:
        continue
    t = train.groupby(col, dropna=False)[y_col].agg(["mean", "count"]).rename(columns={"mean": "default_rate", "count": "n"})
    t = t.sort_values("n", ascending=False)
    print("Tasa de default por", col)
    display(t.head(15))


### Distribuciones `EXT_SOURCE_*` por `TARGET`


### Documentación — Histogramas `EXT_SOURCE_*`

- Para cada `EXT_SOURCE` disponible, superpone histogramas por clase (`TARGET` 0 vs 1) en [0, 1].
- **Densidad** (`density=True`): comparar formas aunque los tamaños de clase difieran.


In [ ]:
ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in train.columns]
if not ext_cols:
    print("No hay EXT_SOURCE en train.")
else:
    fig, axes = plt.subplots(1, len(ext_cols), figsize=(4 * len(ext_cols), 3.5))
    if len(ext_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, ext_cols):
        for tval, label, color in [(0, "TARGET=0", "#4c78a8"), (1, "TARGET=1", "#f58518")]:
            s = train.loc[train[y_col] == tval, col].dropna()
            s = s[(s >= 0) & (s <= 1)]
            ax.hist(s, bins=40, alpha=0.55, label=label, color=color, density=True)
        ax.set_title(col)
        ax.set_xlabel("valor")
        ax.legend(fontsize=8)
    plt.suptitle("EXT_SOURCE (recorte [0,1])", y=1.02)
    plt.tight_layout()


### Tabla `bureau`: filas por cliente

Registros en `bureau.csv` por `SK_ID_CURR` y default por decil (aprox.).


### Documentación — Agregación mínima de `bureau`

- Lee solo **`SK_ID_CURR`** de `bureau.csv` para contar cuántos registros de buró tiene cada cliente.
- **Merge** con train: añade `n_bureau_records` (0 si no hay filas en buró).
- **Deciles** (`qcut`) con `duplicates="drop"` si hay empates: tasa de default por franja de volumen de historial en buró.


In [ ]:
bureau_ids = read_table("bureau", usecols=[k_curr])
n_per = bureau_ids.groupby(k_curr).size().rename("n_bureau_records")
train_nb = train[[k_curr, y_col]].merge(n_per, on=k_curr, how="left")
train_nb["n_bureau_records"] = train_nb["n_bureau_records"].fillna(0)

print(train_nb["n_bureau_records"].describe(percentiles=[0.5, 0.9, 0.99]))
print("Tasa de default por decil de n_bureau_records (aprox.):")
train_nb["decile"] = pd.qcut(train_nb["n_bureau_records"], q=10, duplicates="drop")
display(train_nb.groupby("decile", observed=True)[y_col].agg(["mean", "count"]))


## Auditoría preliminar de leakage (`application_*`)

Revisar nombres y agregaciones temporales en tablas auxiliares en el notebook 02.


### Documentación — Heurística de nombres (leakage)

- Busca subcadenas en mayúsculas (`TARGET`, `DEFAULT`, etc.) en nombres de columnas para revisión manual.
- **`FLAG_DOCUMENT_*`**: cuenta documentos aportados; no implica leakage por sí solo, pero conviene documentar en el informe técnico.


In [ ]:
sus = [
    c
    for c in train.columns
    if any(x in c.upper() for x in ("TARGET", "DEFAULT", "FUTURE", "POST"))
]
print("Columnas a revisar por nombre:", [c for c in sus if c != y_col])

doc_flags = [c for c in train.columns if "FLAG_DOCUMENT" in c]
print("FLAG_DOCUMENT_*:", len(doc_flags), "columnas")


## Reproducibilidad

Valores de `configs/base.yaml`.


### Documentación — Reproducibilidad del pipeline

- Lee **`configs/base.yaml`** y muestra semilla, fracciones de split y métricas previstas.
- **Uso**: trazabilidad entre este EDA y la validación/modelado en otros notebooks.


In [ ]:
base_path = project_root() / "configs" / "base.yaml"
with open(base_path, encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print("seed:", base_cfg.get("seed"))
print("splits:", base_cfg.get("splits"))
print("métricas:", base_cfg.get("metrics"))


## EDA univariado, bivariado y multivariado

### Documentación — EDA univariado

- **Objetivo:** entender la distribución de cada variable por separado (sin condicionar por otras).
- **Qué calculamos aquí:** estadísticos descriptivos para numéricas y frecuencias para categóricas.
- **Por qué importa:** detecta asimetrías, outliers, colas pesadas, categorías raras y variables casi constantes antes de modelar.

In [ ]:
# Univariado: resumen de numéricas
num_stats = train[num_feats].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
num_stats["missing_rate"] = train[num_feats].isna().mean()
num_stats["skew"] = train[num_feats].skew(numeric_only=True)
print("Top 15 variables numéricas por missing_rate:")
display(num_stats.sort_values("missing_rate", ascending=False).head(15))

print("Top 15 variables numéricas por |skew| (asimetría):")
display(num_stats.reindex(num_stats["skew"].abs().sort_values(ascending=False).index).head(15))

# Univariado: resumen de categóricas
cat_summary = []
for col in obj_cols:
    vc = train[col].value_counts(dropna=False)
    top_cat = vc.index[0] if len(vc) else None
    top_freq = float(vc.iloc[0] / len(train)) if len(vc) else np.nan
    cat_summary.append(
        {
            "column": col,
            "n_unique": train[col].nunique(dropna=False),
            "missing_rate": train[col].isna().mean(),
            "top_category": top_cat,
            "top_share": top_freq,
        }
    )
cat_summary = pd.DataFrame(cat_summary).sort_values(["n_unique", "missing_rate"], ascending=[False, False])
print("Resumen univariado categóricas (top 15 por cardinalidad):")
display(cat_summary.head(15))

### Documentación — EDA bivariado

- **Objetivo:** medir relaciones entre pares de variables o variable vs `TARGET`.
- **Qué hacemos:**
  - numérica vs `TARGET` con diferencia de medias estandarizada,
  - categórica vs `TARGET` con tasa de default por categoría,
  - numérica vs numérica con ranking de correlaciones fuertes.
- **Uso:** priorizar features y detectar pares redundantes o altamente acoplados.

In [ ]:
# Bivariado: numérica vs TARGET (efecto estandarizado)
rows_eff = []
for col in num_feats:
    x0 = train.loc[train[y_col] == 0, col].dropna()
    x1 = train.loc[train[y_col] == 1, col].dropna()
    if len(x0) < 10 or len(x1) < 10:
        continue
    std_pooled = np.sqrt((x0.var() + x1.var()) / 2)
    if std_pooled == 0 or np.isnan(std_pooled):
        continue
    eff = (x1.mean() - x0.mean()) / std_pooled
    rows_eff.append({"column": col, "cohens_d_1_vs_0": eff, "mean_t0": x0.mean(), "mean_t1": x1.mean()})

bi_num_target = pd.DataFrame(rows_eff).sort_values("cohens_d_1_vs_0", key=lambda s: s.abs(), ascending=False)
print("Top 15 numéricas por |Cohen's d| frente a TARGET:")
display(bi_num_target.head(15))

# Bivariado: categórica vs TARGET (default rate)
cat_target_rows = []
for col in obj_cols:
    grp = train.groupby(col, dropna=False)[y_col].agg(["mean", "count"]).reset_index()
    grp = grp.rename(columns={"mean": "default_rate", "count": "n"})
    # evitar categorías muy pequeñas para estabilidad
    grp_valid = grp[grp["n"] >= 200]
    if grp_valid.empty:
        continue
    spread = grp_valid["default_rate"].max() - grp_valid["default_rate"].min()
    cat_target_rows.append({"column": col, "n_groups_ge_200": len(grp_valid), "default_rate_spread": spread})

bi_cat_target = pd.DataFrame(cat_target_rows).sort_values("default_rate_spread", ascending=False)
print("Categóricas con mayor dispersión de default_rate (grupos n>=200):")
display(bi_cat_target.head(15))

# Bivariado: numérica vs numérica (colinealidad simple)
corr_num = train[num_feats].corr(numeric_only=True).abs()
upper = corr_num.where(np.triu(np.ones(corr_num.shape), k=1).astype(bool))
pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "var1", "level_1": "var2", 0: "abs_corr"})
    .sort_values("abs_corr", ascending=False)
)
print("Top 20 pares numéricos por correlación absoluta:")
display(pairs.head(20))

### Documentación — EDA multivariado

- **Objetivo:** analizar estructuras conjuntas entre varias variables simultáneamente.
- **Qué hacemos:**
  - imputación simple para análisis exploratorio,
  - estandarización,
  - PCA para ver varianza explicada acumulada,
  - visualización de clientes en el plano PC1–PC2 coloreado por `TARGET`.
- **Interpretación:** no es modelo final; sirve para detectar separabilidad parcial, redundancia y complejidad intrínseca de la señal.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Multivariado: PCA sobre numéricas (sin TARGET)
num_for_pca = [c for c in num_feats if train[c].notna().sum() > 1000]
X = train[num_for_pca].copy()

# Imputación robusta para exploración (mediana)
X = X.fillna(X.median(numeric_only=True))

# Submuestra para costo controlado
if len(X) > 80000:
    idx = X.sample(80000, random_state=42).index
    Xs = X.loc[idx]
    ys = train.loc[idx, y_col]
else:
    Xs = X
    ys = train[y_col]

scaler = StandardScaler()
Xs_scaled = scaler.fit_transform(Xs)

pca = PCA(n_components=min(20, Xs_scaled.shape[1]), random_state=42)
pcs = pca.fit_transform(Xs_scaled)
expl = pca.explained_variance_ratio_
cum_expl = np.cumsum(expl)

pca_table = pd.DataFrame({
    "component": np.arange(1, len(expl) + 1),
    "explained_variance_ratio": expl,
    "cumulative_explained_variance": cum_expl,
})
print("Varianza explicada por PCA (primeras componentes):")
display(pca_table.head(15))

# Gráfico acumulado
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pca_table["component"], pca_table["cumulative_explained_variance"], marker="o")
ax.set_xlabel("Número de componentes")
ax.set_ylabel("Varianza explicada acumulada")
ax.set_title("PCA — varianza explicada acumulada")
ax.set_ylim(0, 1.01)
plt.tight_layout()

# Plano PC1-PC2 con muestra balanceada para visual
pc_df = pd.DataFrame({"PC1": pcs[:, 0], "PC2": pcs[:, 1], "TARGET": ys.values})
plot_df = pd.concat([
    pc_df[pc_df["TARGET"] == 0].sample(min(3000, (pc_df["TARGET"] == 0).sum()), random_state=42),
    pc_df[pc_df["TARGET"] == 1].sample(min(3000, (pc_df["TARGET"] == 1).sum()), random_state=42),
])

fig, ax = plt.subplots(figsize=(6.5, 5))
for tval, color, label in [(0, "#4c78a8", "TARGET=0"), (1, "#f58518", "TARGET=1")]:
    ss = plot_df[plot_df["TARGET"] == tval]
    ax.scatter(ss["PC1"], ss["PC2"], s=8, alpha=0.35, c=color, label=label)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA (PC1 vs PC2) por TARGET")
ax.legend()
plt.tight_layout()

## EDA premium (estadística avanzada)

Esta sección profundiza en evidencia estadística y estructura global del dataset: tests no paramétricos, información mutua, colinealidad (VIF) y proyecciones no lineales.

### Documentación — Univariado premium (numéricas vs TARGET)

- Usamos **Mann-Whitney U** por variable numérica para contrastar si la distribución en `TARGET=1` difiere de `TARGET=0` sin asumir normalidad.
- Reportamos `p_value` y `-log10(p)` para ranking.
- Añadimos **q-value aproximado (Benjamini-Hochberg)** para controlar falsos positivos por múltiples pruebas.

In [ ]:
from scipy.stats import mannwhitneyu

mw_rows = []
for col in num_feats:
    x0 = train.loc[train[y_col] == 0, col].dropna()
    x1 = train.loc[train[y_col] == 1, col].dropna()
    if len(x0) < 30 or len(x1) < 30:
        continue
    try:
        stat, p = mannwhitneyu(x1, x0, alternative="two-sided")
    except ValueError:
        continue
    mw_rows.append({"column": col, "u_stat": stat, "p_value": p})

mw = pd.DataFrame(mw_rows)
if not mw.empty:
    mw = mw.sort_values("p_value", ascending=True).reset_index(drop=True)
    # Benjamini-Hochberg q-value aproximado
    m = len(mw)
    mw["rank"] = np.arange(1, m + 1)
    mw["q_bh"] = (mw["p_value"] * m / mw["rank"]).clip(upper=1.0)
    mw["minus_log10_p"] = -np.log10(mw["p_value"].clip(lower=1e-300))
    print("Top 20 variables numéricas por evidencia de diferencia de distribución (Mann-Whitney):")
    display(mw.head(20))
else:
    print("No se pudieron calcular pruebas Mann-Whitney con el mínimo de muestras definido.")

### Documentación — Bivariado premium (señal predictiva y redundancia)

- **Mutual Information (MI)** con `TARGET`: capta dependencia no lineal de variables numéricas con la etiqueta.
- **VIF** (Variance Inflation Factor) en un subconjunto de numéricas para detectar multicolinealidad alta (`VIF > 10` suele ser señal fuerte de redundancia).
- Esto ayuda a decidir selección de features o regularización para modelos lineales.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Mutual Information (numéricas -> TARGET)
mi_cols = [c for c in num_feats if train[c].notna().mean() > 0.6]
X_mi = train[mi_cols].copy().fillna(train[mi_cols].median(numeric_only=True))
y_mi = train[y_col].astype(int)

# Submuestra por costo
if len(X_mi) > 120000:
    idx_mi = X_mi.sample(120000, random_state=42).index
    X_mi = X_mi.loc[idx_mi]
    y_mi = y_mi.loc[idx_mi]

mi = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_df = pd.DataFrame({"column": mi_cols, "mutual_info": mi}).sort_values("mutual_info", ascending=False)
print("Top 20 numéricas por Mutual Information con TARGET:")
display(mi_df.head(20))

# VIF: usar top-20 por MI para limitar costo y estabilidad
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    vif_cols = mi_df.head(20)["column"].tolist()
    X_vif = train[vif_cols].copy().fillna(train[vif_cols].median(numeric_only=True))
    # Estandarizar evita escalas extremas en inversión de matriz
    X_vif = (X_vif - X_vif.mean()) / (X_vif.std(ddof=0) + 1e-12)
    X_vif_np = X_vif.to_numpy()
    vif_vals = []
    for i, c in enumerate(vif_cols):
        vif_vals.append({"column": c, "VIF": variance_inflation_factor(X_vif_np, i)})
    vif_df = pd.DataFrame(vif_vals).sort_values("VIF", ascending=False)
    print("VIF sobre top-20 MI (colinealidad):")
    display(vif_df)
except Exception as e:
    print("VIF omitido (statsmodels no disponible o error numérico):", e)

### Documentación — Multivariado premium (embeddings no lineales)

- Proyectamos una submuestra a 2D con **UMAP** (si está instalado) o **t-SNE** (fallback con sklearn).
- El objetivo no es predecir, sino inspeccionar si hay estructura global y separación parcial entre clases en un espacio no lineal.
- Estas figuras son exploratorias y sensibles a hiperparámetros (vecinos, perplexity, tamaño de muestra).

In [ ]:
from sklearn.manifold import TSNE

embed_cols = mi_df.head(25)["column"].tolist() if "mi_df" in globals() else num_for_pca[:25]
X_emb = train[embed_cols].copy().fillna(train[embed_cols].median(numeric_only=True))
y_emb = train[y_col].astype(int)

# Muestra controlada para visualización
n0 = min(2500, int((y_emb == 0).sum()))
n1 = min(2500, int((y_emb == 1).sum()))
idx0 = X_emb[y_emb == 0].sample(n0, random_state=42).index
idx1 = X_emb[y_emb == 1].sample(n1, random_state=42).index
idx = idx0.union(idx1)
X_plot = X_emb.loc[idx]
y_plot = y_emb.loc[idx]

# Escalado robusto
X_plot = (X_plot - X_plot.median()) / (X_plot.quantile(0.75) - X_plot.quantile(0.25) + 1e-12)

coords = None
method_used = None
try:
    import umap

    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="euclidean", random_state=42)
    coords = reducer.fit_transform(X_plot)
    method_used = "UMAP"
except Exception:
    tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=42)
    coords = tsne.fit_transform(X_plot)
    method_used = "t-SNE"

emb_df = pd.DataFrame({"dim1": coords[:, 0], "dim2": coords[:, 1], "TARGET": y_plot.values})

fig, ax = plt.subplots(figsize=(6.8, 5.2))
for tval, color, label in [(0, "#4c78a8", "TARGET=0"), (1, "#f58518", "TARGET=1")]:
    ss = emb_df[emb_df["TARGET"] == tval]
    ax.scatter(ss["dim1"], ss["dim2"], s=8, alpha=0.35, c=color, label=label)
ax.set_title(f"{method_used} 2D (submuestra balanceada)")
ax.set_xlabel("dim1")
ax.set_ylabel("dim2")
ax.legend()
plt.tight_layout()

print("Método usado:", method_used, "| filas proyectadas:", len(emb_df), "| columnas:", len(embed_cols))

## Resumen y siguientes pasos

| Hallazgo | Implicación |
|----------|-------------|
| Clase minoritaria | Estratificar CV; PR-AUC y calibración |
| Nulos por bloques | Imputación / NaN-friendly |
| NA distinto entre TARGET 0/1 | Missing como señal |
| KS train vs test alto | Deriva de datos |
| Tablas auxiliares largas | Agregaciones por cliente + tiempo |
| EXT_SOURCE / DAYS_* con corr | Features fuertes |
| Cramér V alto | Encoding / árboles |
| Mann-Whitney/MI con señal fuerte | Priorizar variables robustas para baseline lineal y boosting |
| VIF alto en top variables | Reducir redundancia (drop/regularizar) antes de modelos lineales |
| UMAP/t-SNE con solapamiento parcial | Frontera no lineal; calibración y thresholding serán clave |

Siguiente: `02_feature_engineering.ipynb`.
